# Explainable ML Framework for Early GDM Prediction Without OGTT

**Research implementation notebook** accompanying the co-authored paper in `paper/`.

> Research use only. This notebook is not a medical device and must not be used for diagnosis or treatment decisions.

## Before running
1. Obtain the dataset from its original Kaggle source (see the paper's reference [19]); the dataset is intentionally not redistributed in this repository.
2. In Google Colab, place the workbook at the configured Google Drive path or update `DATA_PATH`.
3. Set `GEMINI_API_KEY` as a Colab secret/environment variable before running the report-extraction section. Never paste API keys into notebook cells.
4. Run sections in order. The notebook has been cleared of saved outputs to avoid publishing patient rows, credentials, or stale runtime state.

The notebook preserves the research workflow. See `docs/reproducibility-and-limitations.md` for important implementation caveats, including preprocessing-before-split leakage risk and the distinction between paper-reported results and external validation.


In [ ]:

!pip install -q xgboost lightgbm catboost shap imbalanced-learn scikit-learn --upgrade
!pip uninstall -y scikit-learn imbalanced-learn
!pip install -q scikit-learn==1.6.1 imbalanced-learn==0.13.0

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import os, json, random, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
)
from imblearn.over_sampling import SMOTE
import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/GDM_Project"  # Update this path for your own Colab Drive
os.makedirs(PROJECT_DIR, exist_ok=True)

DATA_PATH = f"{PROJECT_DIR}/Gestational Diabetic Dat Set.xlsx"
TARGET = "GDM"

In [ ]:

NUMERIC = ["Age", "Num_Pregnancies", "Gestation_Weeks", "BMI", "HDL",
           "Systolic_BP", "Diastolic_BP", "Hemoglobin", "OGTT"]
BINARY = ["Family_History", "Unexplained_Prenatal_Loss", "Large_Child_History",
          "PCOS", "Sedentary_Lifestyle", "Prediabetes"]
ALL_FEATURES = NUMERIC + BINARY

COLUMN_MAP = {
    "Case Number": "Case_Number", "Age": "Age", "No of Pregnancy": "Num_Pregnancies",
    "Gestation in previous Pregnancy": "Gestation_Weeks", "BMI": "BMI", "HDL": "HDL",
    "Family History": "Family_History", "unexplained prenetal loss": "Unexplained_Prenatal_Loss",
    "Large Child or Birth Default": "Large_Child_History", "PCOS": "PCOS",
    "Sys BP": "Systolic_BP", "Dia BP": "Diastolic_BP", "OGTT": "OGTT",
    "Hemoglobin": "Hemoglobin", "Sedentary Lifestyle": "Sedentary_Lifestyle",
    "Prediabetes": "Prediabetes", "Class Label(GDM /Non GDM)": "GDM",
}

In [ ]:

def load_dataset(path, column_map=None, drop_cols=None):
    if not os.path.exists(path):
        raise FileNotFoundError(f"can't find {path} - upload it or fix DATA_PATH")
    df = pd.read_excel(path)
    if column_map:
        df = df.rename(columns=column_map)
    if drop_cols:
        df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    print(f"loaded {df.shape[0]} rows, {df.shape[1]} cols")
    return df

df_raw = load_dataset(DATA_PATH, COLUMN_MAP, drop_cols=["Case_Number"])
display(df_raw.head())
print(df_raw.dtypes)
display(df_raw.describe().T)

### EDA

In [ ]:
missing = df_raw.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("missing values:\n", missing)

counts = df_raw[TARGET].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x=TARGET, data=df_raw, ax=ax[0], palette="Set2")
ax[1].pie(counts, labels=["No GDM", "GDM"], autopct="%1.1f%%", colors=sns.color_palette("Set2"))
plt.tight_layout(); plt.show()
print("imbalance ratio:", round(counts.min() / counts.max(), 2))

In [ ]:
plt.figure(figsize=(11, 9))
corr = df_raw[ALL_FEATURES + [TARGET]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("feature correlations vs GDM")
plt.tight_layout(); plt.show()
print(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, feat in enumerate(NUMERIC):
    sns.histplot(df_raw, x=feat, hue=TARGET, kde=True, ax=axes[i], element="step", stat="density")
for j in range(len(NUMERIC), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()
for i, feat in enumerate(NUMERIC):
    sns.boxplot(data=df_raw, x=TARGET, y=feat, ax=axes[i])
for j in range(len(NUMERIC), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.violinplot(data=df_raw, x=TARGET, y="BMI", ax=axes[0, 0])
sns.violinplot(data=df_raw, x=TARGET, y="Age", ax=axes[0, 1])
fam_pcos = df_raw.groupby(TARGET)[["Family_History", "PCOS"]].mean().reset_index().melt(id_vars=TARGET)
sns.barplot(data=fam_pcos, x="variable", y="value", hue=TARGET, ax=axes[1, 0])
sns.violinplot(data=df_raw, x=TARGET, y="OGTT", ax=axes[1, 1])
plt.tight_layout(); plt.show()

In [ ]:

def clean(df):
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"dropped {before - len(df)} dupes")

    df[NUMERIC] = SimpleImputer(strategy="median").fit_transform(df[NUMERIC])
    df[BINARY] = SimpleImputer(strategy="most_frequent").fit_transform(df[BINARY])

    for col in NUMERIC:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        df[col] = df[col].clip(lo, hi)
    return df

df_clean = clean(df_raw)

In [ ]:
#  Feature engineering
def engineer(df):
    df = df.copy()
    df["BMI_Category_Code"] = pd.cut(df["BMI"], [0, 18.5, 25, 30, 100]).cat.codes
    df["BP_Category_Code"] = df.apply(
        lambda r: 0 if r.Systolic_BP < 120 and r.Diastolic_BP < 80
        else 1 if r.Systolic_BP < 130 and r.Diastolic_BP < 80
        else 2 if r.Systolic_BP < 140 or r.Diastolic_BP < 90 else 3, axis=1
    )
    df["Age_Group_Code"] = pd.cut(df["Age"], [0, 20, 25, 30, 35, 100]).cat.codes
    df["Risk_Score"] = (
        (df.BMI >= 30) * 2 + (df.Age >= 35) * 2 + df.Family_History * 2 + df.PCOS * 1.5
        + df.Prediabetes * 2 + df.Unexplained_Prenatal_Loss + df.Large_Child_History
        + df.Sedentary_Lifestyle + (df.Systolic_BP >= 130)
    )
    return df

df_eng = engineer(df_clean)
ENGINEERED = ["BMI_Category_Code", "BP_Category_Code", "Age_Group_Code", "Risk_Score"]
FEATURES_ALL = ALL_FEATURES + ENGINEERED
FEATURES_NO_OGTT = [f for f in FEATURES_ALL if f != "OGTT"]
RAW_FEATURES_NO_OGTT = [f for f in ALL_FEATURES if f != "OGTT"]

plt.figure()
sns.boxplot(data=df_eng, x=TARGET, y="Risk_Score")
plt.title("risk score by outcome")
plt.show()

In [ ]:

def split_scale_smote(df, features, target, test_size=0.2):
    X, y = df[features], df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=RANDOM_STATE
    )
    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=features, index=X_train.index)
    X_test = pd.DataFrame(scaler.transform(X_test), columns=features, index=X_test.index)

    if y_train.value_counts(normalize=True).min() < 0.4:
        X_train, y_train = SMOTE(random_state=RANDOM_STATE).fit_resample(X_train, y_train)

    return X_train, X_test, y_train, y_test, scaler

X_tr, X_te, y_tr, y_te, scaler = split_scale_smote(df_eng, FEATURES_NO_OGTT, TARGET)

In [ ]:

MODEL_GRIDS = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {"C": [0.01, 0.1, 1, 5, 10], "solver": ["lbfgs", "liblinear"]}),
    "Random Forest": (RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": [200, 300, 400], "max_depth": [4, 6, 8, None], "min_samples_leaf": [1, 2, 4]}),
    "Extra Trees": (ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": [200, 300, 400], "max_depth": [4, 6, 8, None]}),
    "XGBoost": (XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss", n_jobs=-1),
        {"n_estimators": [200, 300], "max_depth": [3, 4, 5, 6], "learning_rate": [0.01, 0.05, 0.1]}),
    "LightGBM": (LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
        {"n_estimators": [200, 300], "num_leaves": [15, 31, 63], "learning_rate": [0.01, 0.05, 0.1]}),
    "CatBoost": (CatBoostClassifier(random_state=RANDOM_STATE, verbose=0),
        {"iterations": [200, 300], "depth": [4, 6, 8], "learning_rate": [0.01, 0.05, 0.1]}),
}

def tune(estimator, params, X_train, y_train, n_iter=25):
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(estimator, params, n_iter=n_iter, scoring="roc_auc",
                                 cv=cv, random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train, y_train)
    return search.best_estimator_, search.best_score_

def evaluate(name, model, X_test, y_test, cv_score):
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "CV ROC-AUC": cv_score,
    }, pred, proba

In [ ]:

trained, cv_scores, results, preds = {}, {}, [], {}
for name, (est, params) in MODEL_GRIDS.items():
    print(f"tuning {name}...")
    model, score = tune(est, params, X_tr, y_tr)
    trained[name] = model
    cv_scores[name] = score
    metrics, pred, proba = evaluate(name, model, X_te, y_te, score)
    results.append(metrics)
    preds[name] = (pred, proba)

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
display(results_df)
best_name = results_df.iloc[0]["Model"]
best_model = trained[best_name]
print("best model:", best_name)

tree_based = any(k in best_name for k in ["Forest", "Trees", "XGBoost", "LightGBM", "CatBoost"])
explainer = shap.TreeExplainer(best_model) if tree_based else shap.LinearExplainer(best_model, X_tr)

### Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
for i, name in enumerate(trained):
    cm = confusion_matrix(y_te, preds[name][0])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i],
                xticklabels=["No GDM", "GDM"], yticklabels=["No GDM", "GDM"])
    axes[i].set_title(name)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 7))
for name in trained:
    fpr, tpr, _ = roc_curve(y_te, preds[name][1])
    plt.plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_te, preds[name][1]):.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.legend(); plt.title("ROC curves"); plt.show()

plt.figure(figsize=(8, 7))
for name in trained:
    p, r, _ = precision_recall_curve(y_te, preds[name][1])
    plt.plot(r, p, label=name)
plt.legend(); plt.title("precision-recall curves"); plt.show()

print(classification_report(y_te, preds[best_name][0], target_names=["No GDM", "GDM"]))

In [ ]:
shap_vals = explainer.shap_values(X_te)
shap_vals = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

shap.summary_plot(shap_vals, X_te, plot_type="bar", show=False); plt.show()
shap.summary_plot(shap_vals, X_te, show=False); plt.show()

In [ ]:

idx = 0

base = explainer.expected_value
if isinstance(base, (list, np.ndarray)):
    base = base[1]

exp = shap.Explanation(
    values=shap_vals[idx, :, 1] if np.array(shap_vals).ndim == 3 else shap_vals[idx],
    base_values=base,
    data=X_te.iloc[idx].values,
    feature_names=X_te.columns.tolist()
)

shap.plots.waterfall(exp)
plt.show()


print(
    "Predicted Probability:",
    best_model.predict_proba(X_te.iloc[[idx]])[0, 1],
    "Actual:",
    y_te.iloc[idx]
)

### With-OGTT vs without-OGTT experiment

In [ ]:
def run_experiment(features, label):
    Xtr, Xte, ytr, yte, _ = split_scale_smote(df_eng, features, TARGET)
    rows = {}
    for name, (est, params) in MODEL_GRIDS.items():
        model, cv = tune(est, params, Xtr, ytr, n_iter=15)
        metrics, _, _ = evaluate(name, model, Xte, yte, cv)
        metrics["Experiment"] = label
        rows[name] = metrics
    return pd.DataFrame(rows.values())

exp_with = run_experiment(FEATURES_ALL, "With OGTT")
exp_without = run_experiment(FEATURES_NO_OGTT, "Without OGTT")

combined = pd.concat([exp_with, exp_without])
pivot = combined.pivot(index="Model", columns="Experiment", values="ROC-AUC")
display(pivot)

pivot.plot(kind="bar", figsize=(10, 6))
plt.title("ROC-AUC Comparison")
plt.ylabel("ROC-AUC")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
with open(f"{PROJECT_DIR}/best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
with open(f"{PROJECT_DIR}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
with open(f"{PROJECT_DIR}/explainer.pkl", "wb") as f:
    pickle.dump(explainer, f)
with open(f"{PROJECT_DIR}/meta.pkl", "wb") as f:
    pickle.dump({
        "tree_based": tree_based,
        "FEATURES_NO_OGTT": FEATURES_NO_OGTT,
        "RAW_FEATURES_NO_OGTT": RAW_FEATURES_NO_OGTT,
        "best_name": best_name,
    }, f)

print("Saved to", PROJECT_DIR)

import shutil
src = "/content/Gestational Diabetic Dat Set.xlsx"
if os.path.exists(src):
    shutil.copy(src, f"{PROJECT_DIR}/Gestational Diabetic Dat Set.xlsx")
    print("Dataset copied to Drive too.")

---
# SECTION B: APP (run every session)
This is all you need on a normal day. Run every cell in this section top to bottom, then use the app that appears at the bottom.

In [ ]:

!pip install -q google-genai pdf2image
!apt-get -qq update && apt-get -qq install -y poppler-utils

In [ ]:

import os, json, re, pickle
import numpy as np
import pandas as pd
import shap
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive, files
from google import genai
from pdf2image import convert_from_path
from PIL import Image

In [ ]:

drive.mount('/content/drive')
PROJECT_DIR = "/content/drive/MyDrive/GDM_Project"  # Update this path for your own Colab Drive

with open(f"{PROJECT_DIR}/best_model.pkl", "rb") as f:
    best_model = pickle.load(f)
with open(f"{PROJECT_DIR}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
with open(f"{PROJECT_DIR}/explainer.pkl", "rb") as f:
    explainer = pickle.load(f)
with open(f"{PROJECT_DIR}/meta.pkl", "rb") as f:
    meta = pickle.load(f)

tree_based = meta["tree_based"]
FEATURES_NO_OGTT = meta["FEATURES_NO_OGTT"]
RAW_FEATURES_NO_OGTT = meta["RAW_FEATURES_NO_OGTT"]
best_name = meta["best_name"]

print("Loaded model:", best_name)

In [ ]:

NUMERIC = ["Age", "Num_Pregnancies", "Gestation_Weeks", "BMI", "HDL",
           "Systolic_BP", "Diastolic_BP", "Hemoglobin", "OGTT"]
BINARY = ["Family_History", "Unexplained_Prenatal_Loss", "Large_Child_History",
          "PCOS", "Sedentary_Lifestyle", "Prediabetes"]
ALL_FEATURES = NUMERIC + BINARY

In [ ]:

def engineer(df):
    df = df.copy()
    df["BMI_Category_Code"] = pd.cut(df["BMI"], [0, 18.5, 25, 30, 100]).cat.codes
    df["BP_Category_Code"] = df.apply(
        lambda r: 0 if r.Systolic_BP < 120 and r.Diastolic_BP < 80
        else 1 if r.Systolic_BP < 130 and r.Diastolic_BP < 80
        else 2 if r.Systolic_BP < 140 or r.Diastolic_BP < 90 else 3, axis=1
    )
    df["Age_Group_Code"] = pd.cut(df["Age"], [0, 20, 25, 30, 35, 100]).cat.codes
    df["Risk_Score"] = (
        (df.BMI >= 30) * 2 + (df.Age >= 35) * 2 + df.Family_History * 2 + df.PCOS * 1.5
        + df.Prediabetes * 2 + df.Unexplained_Prenatal_Loss + df.Large_Child_History
        + df.Sedentary_Lifestyle + (df.Systolic_BP >= 130)
    )
    return df

In [ ]:

def predict_gdm(patient, model, scaler, explainer, features, tree_based=True):
    row = engineer(pd.DataFrame([patient]))

    missing = [f for f in features if f not in row.columns]
    if missing:
        raise ValueError(f"Missing features: {missing}")

    X = pd.DataFrame(scaler.transform(row[features]), columns=features)
    prob = float(model.predict_proba(X)[0, 1])

    sv = explainer.shap_values(X)
    if isinstance(sv, list):
        sv = sv[1]
    elif len(np.array(sv).shape) == 3:
        sv = sv[:, :, 1]
    sv = sv[0]

    contrib = pd.Series(sv, index=features).sort_values(key=np.abs, ascending=False)

    return {
        "probability_gdm": round(prob, 4),
        "predicted_class": "Yes" if prob >= 0.5 else "No",
        "top_contributing_features": [
            {"feature": f, "shap_value": round(float(v), 4)}
            for f, v in contrib.head(5).items()
        ],
    }

In [ ]:


sample_patient = {
    "Age": 33, "Num_Pregnancies": 2, "Gestation_Weeks": 14, "BMI": 31.2, "HDL": 42,
    "Family_History": 1, "Unexplained_Prenatal_Loss": 0, "Large_Child_History": 0,
    "PCOS": 1, "Systolic_BP": 128, "Diastolic_BP": 82, "Hemoglobin": 11.0,
    "Sedentary_Lifestyle": 1, "Prediabetes": 0,
}
print(json.dumps(predict_gdm(sample_patient, best_model, scaler, explainer, FEATURES_NO_OGTT, tree_based), indent=2))

In [ ]:

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY in your environment before using report extraction.")
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL_ID = "gemini-3.1-flash-lite"

In [ ]:

EXTRACTABLE_FIELDS = ["Age", "BMI", "HDL", "Hemoglobin",
                       "Systolic_BP", "Diastolic_BP", "Gestation_Weeks"]
MANUAL_ONLY_FIELDS = [f for f in RAW_FEATURES_NO_OGTT if f not in EXTRACTABLE_FIELDS]

EXTRACTION_PROMPT = f"""
You are a medical data extraction assistant.
Read the attached medical report image(s) carefully.
Extract ONLY these fields: {EXTRACTABLE_FIELDS}
Return ONLY valid JSON, no markdown fences, no explanation.
If a value is not present, set it to null. Do not guess.
Schema: {{{", ".join(f'"{f}": null' for f in EXTRACTABLE_FIELDS)}}}
"""

def file_to_images(path):
    if path.lower().endswith(".pdf"):
        return convert_from_path(path, dpi=200)
    return [Image.open(path)]

def extract_fields_from_report(path):
    images = file_to_images(path)
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[EXTRACTION_PROMPT, *images],
    )
    text = re.sub(r"^```json|```$", "", response.text.strip(), flags=re.MULTILINE).strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        raise ValueError(f"Gemini output wasn't valid JSON:\n{text}")
    return {k: data.get(k) for k in EXTRACTABLE_FIELDS}

In [ ]:

VALID_RANGES = {
    "Age": (10, 60), "Num_Pregnancies": (0, 15), "Gestation_Weeks": (1, 42),
    "BMI": (10, 70), "HDL": (5, 150), "Systolic_BP": (60, 250),
    "Diastolic_BP": (40, 150), "Hemoglobin": (3, 20),
}

def validate_patient(patient):
    errors = []
    for field, (lo, hi) in VALID_RANGES.items():
        val = patient.get(field)
        if val is not None and not (lo <= float(val) <= hi):
            errors.append(f"{field}={val} outside plausible range [{lo}, {hi}]")
    for field in BINARY:
        val = patient.get(field)
        if val is not None and int(val) not in (0, 1):
            errors.append(f"{field}={val} must be 0 or 1")
    return errors

In [ ]:

FIELD_LABELS = {
    "Age": "Age (years)", "Num_Pregnancies": "Previous pregnancies",
    "Gestation_Weeks": "Gestation (weeks)", "BMI": "BMI", "HDL": "HDL (mg/dL)",
    "Systolic_BP": "Systolic BP", "Diastolic_BP": "Diastolic BP",
    "Hemoglobin": "Hemoglobin (g/dL)",
    "Family_History": "Family history of diabetes",
    "Unexplained_Prenatal_Loss": "Unexplained prenatal loss history",
    "Large_Child_History": "History of large birth-weight child",
    "PCOS": "PCOS", "Sedentary_Lifestyle": "Sedentary lifestyle",
    "Prediabetes": "Prediabetes",
}

def build_field_widget(field, value=None):
    if field in BINARY:
        w = widgets.ToggleButtons(
            options=[("No", 0), ("Yes", 1)],
            description=FIELD_LABELS[field],
            value=0,
        )
        if value is not None:
            w.value = int(value)
        return w
    w = widgets.FloatText(description=FIELD_LABELS[field], value=0.0)
    if value is not None:
        w.value = float(value)
    return w

def build_form(fields, known_values=None):
    known_values = known_values or {}
    return {f: build_field_widget(f, known_values.get(f)) for f in fields}

def collect_patient(form):
    patient = {}
    for f, w in form.items():
        if f in BINARY:
            patient[f] = int(w.value)
        else:
            patient[f] = float(w.value)
    return patient

In [ ]:

def run_prediction(patient):
    errors = validate_patient(patient)
    if errors:
        print("❌ Validation failed:")
        for e in errors:
            print(" -", e)
        return

    print("Patient input being used:")
    print(json.dumps(patient, indent=2))

    result = predict_gdm(patient, best_model, scaler, explainer, FEATURES_NO_OGTT, tree_based)
    print("\nPrediction result:")
    print(json.dumps(result, indent=2))

    summary_prompt = f"""
You are assisting obstetricians. Summarize this prediction in simple medical language.
Prediction: {result['predicted_class']}
Probability: {result['probability_gdm']*100:.1f}%
Top contributing features: {[c['feature'] for c in result['top_contributing_features']]}
Explain the model probability and important contributing factors. Avoid assigning unvalidated risk bands or making a diagnosis. Recommend clinician review and appropriate standard testing. Max 150 words.
"""

    try:
        summary = client.models.generate_content(model=MODEL_ID, contents=summary_prompt).text
    except Exception as e:
        summary = f"(AI summary unavailable right now: {e})"

    print("\n AI Clinical Summary:\n", summary)
    return result, summary

In [ ]:

def launch_gdm_app():
    mode = widgets.ToggleButtons(options=["Manual Entry", "Upload Report (PDF/Image)"])
    display(mode)

    def on_mode_change(change):
        clear_output(wait=True)
        display(mode)

        if mode.value == "Manual Entry":
            form = build_form(RAW_FEATURES_NO_OGTT)
            btn = widgets.Button(description="Predict", button_style="success")
            display(*form.values(), btn)
            btn.on_click(lambda _: run_prediction(collect_patient(form)))
        else:
            upload_btn = widgets.Button(description="Upload report file")
            display(upload_btn)

            def on_upload(_):
                print("Select a PDF or image...")
                uploaded = files.upload()
                path = list(uploaded.keys())[0]
                print("Extracting fields with Gemini...")
                extracted = extract_fields_from_report(path)
                print("Extracted:", extracted)
                print("\nConfirm extracted values and fill the rest:")
                form = build_form(RAW_FEATURES_NO_OGTT, known_values=extracted)
                btn = widgets.Button(description="Predict", button_style="success")
                display(*form.values(), btn)
                btn.on_click(lambda _: run_prediction(collect_patient(form)))
            upload_btn.on_click(on_upload)

    mode.observe(on_mode_change, names="value")

launch_gdm_app()

In [ ]:
import shutil, os

PROJECT_DIR = "/content/drive/MyDrive/GDM_Project"  # Update this path for your own Colab Drive
PACKAGE_DIR = "/content/GDM_model_files"
os.makedirs(PACKAGE_DIR, exist_ok=True)

for fname in ["best_model.pkl", "scaler.pkl", "explainer.pkl", "meta.pkl"]:
    shutil.copy(f"{PROJECT_DIR}/{fname}", f"{PACKAGE_DIR}/{fname}")

shutil.make_archive("/content/GDM_model_package", "zip", PACKAGE_DIR)
print("Created zip at /content/GDM_model_package.zip")

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import io, time, json, re
from sklearn.metrics import confusion_matrix

In [ ]:
test_patients_raw = df_eng.loc[X_te.index, RAW_FEATURES_NO_OGTT].copy()
print(f"{len(test_patients_raw)} test-set patients available for validation")

In [ ]:
def generate_synthetic_report(patient_values, patient_id, template=0):
    img = Image.new("RGB", (600, 500), color="white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 16)
        font_bold = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 18)
    except Exception:
        font = ImageFont.load_default()
        font_bold = font

    field_labels = {
        "Age": "Age (yrs)", "BMI": "BMI (kg/m2)", "HDL": "HDL Cholesterol (mg/dL)",
        "Hemoglobin": "Hemoglobin (g/dL)", "Systolic_BP": "Systolic BP (mmHg)",
        "Diastolic_BP": "Diastolic BP (mmHg)", "Gestation_Weeks": "Gestation (weeks)",
    }

    y = 20
    if template == 0:
        draw.text((20, y), f"Antenatal Lab Report - Patient #{patient_id}", font=font_bold, fill="black")
        y += 40
        for field, label in field_labels.items():
            draw.text((20, y), f"{label}: {patient_values[field]}", font=font, fill="black")
            y += 30
    else:
        draw.text((20, y), "MATERNITY CARE CENTER", font=font_bold, fill="black")
        y += 25
        draw.text((20, y), f"Report ID: {patient_id}", font=font, fill="black")
        y += 35
        draw.line((20, y, 580, y), fill="black")
        y += 15
        for field, label in field_labels.items():
            draw.text((20, y), f"{label:.<35}{patient_values[field]}", font=font, fill="black")
            y += 28

    return img

In [ ]:
N_SAMPLES = 50

sample_ids = test_patients_raw.sample(
    n=min(N_SAMPLES, len(test_patients_raw)), random_state=RANDOM_STATE
).index

validation_rows = []

for i, pid in enumerate(sample_ids):
    true_patient = test_patients_raw.loc[pid].to_dict()
    report_img = generate_synthetic_report(true_patient, pid, template=i % 2)

    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=[EXTRACTION_PROMPT, report_img],
        )
        text = re.sub(r"^```json|```$", "", response.text.strip(), flags=re.MULTILINE).strip()
        extracted = json.loads(text)
    except Exception as e:
        print(f"[{i}] extraction failed for patient {pid}: {e}")
        continue

    llm_patient = dict(true_patient)
    for f in EXTRACTABLE_FIELDS:
        if extracted.get(f) is not None:
            llm_patient[f] = extracted[f]

    true_result = predict_gdm(true_patient, best_model, scaler, explainer, FEATURES_NO_OGTT, tree_based)
    llm_result = predict_gdm(llm_patient, best_model, scaler, explainer, FEATURES_NO_OGTT, tree_based)

    row = {
        "patient_id": pid,
        "true_proba": true_result["probability_gdm"],
        "llm_proba": llm_result["probability_gdm"],
        "true_class": true_result["predicted_class"],
        "llm_class": llm_result["predicted_class"],
    }
    for f in EXTRACTABLE_FIELDS:
        row[f"true_{f}"] = true_patient[f]
        row[f"llm_{f}"] = extracted.get(f)
    validation_rows.append(row)

    if (i + 1) % 10 == 0:
        print(f"  processed {i + 1}/{len(sample_ids)}")
    time.sleep(3)  # be gentle with API rate limits

validation_df = pd.DataFrame(validation_rows)
print(f"\nSuccessfully validated {len(validation_df)} / {len(sample_ids)} patients")

In [ ]:
print(f"{'Field':<15}{'Success Rate':<15}{'MAE':<10}{'Exact Match (rounded)'}")
for f in EXTRACTABLE_FIELDS:
    true_col, llm_col = f"true_{f}", f"llm_{f}"
    mask = validation_df[llm_col].notna()
    success_rate = mask.mean()
    mae = (validation_df.loc[mask, true_col] - validation_df.loc[mask, llm_col]).abs().mean()
    exact_match = (
        validation_df.loc[mask, true_col].round(1) == validation_df.loc[mask, llm_col].round(1)
    ).mean()
    print(f"{f:<15}{success_rate:<15.1%}{mae:<10.2f}{exact_match:.1%}")

In [ ]:
class_agreement = (validation_df["true_class"] == validation_df["llm_class"]).mean()
proba_mae = (validation_df["true_proba"] - validation_df["llm_proba"]).abs().mean()
proba_corr = validation_df["true_proba"].corr(validation_df["llm_proba"])

print(f"Prediction class agreement:                     {class_agreement:.1%}")
print(f"Mean absolute difference in predicted probability: {proba_mae:.4f}")
print(f"Pearson correlation (true vs LLM-derived proba):   {proba_corr:.4f}")

In [ ]:
print(f"{'Field':<15}{'MAE (full precision)'}")
for f in EXTRACTABLE_FIELDS:
    true_col, llm_col = f"true_{f}", f"llm_{f}"
    mask = validation_df[llm_col].notna()
    mae = (validation_df.loc[mask, true_col] - validation_df.loc[mask, llm_col]).abs().mean()
    print(f"{f:<15}{mae:.8f}")